# 5. Calculating QoM Features

Based on the extrema detected in Notebook 4, we calculate the following for
each participant and each movement:
- **QoM_rate** = QoM / duration_in_seconds (extrema per second)

We calculate the features on **4 aggregated signals per participant**:
   - `hands_horizontal` (left + right horizontal)
   - `hands_vertical` (left + right vertical)
   - `head_horizontal` 
   - `head_vertical` 

## 5.1 Imports et configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


PROJECT_DIR = Path.cwd().parent.parent

from config import (
    DATA_DIR, MOVEMENT_SIGNALS, FPS,
)

print(f"FPS : {FPS}")
print(f"Mouvements ({len(MOVEMENT_SIGNALS)}) : {MOVEMENT_SIGNALS}")

FPS : 30.0
Mouvements (6) : ['left_hand_vertical', 'right_hand_vertical', 'left_hand_horizontal', 'right_hand_horizontal', 'head_horizontal', 'head_vertical']


## 5.2 Data Loading

We load:
- the **normalized movements** (notebook 3): to retrieve the
  duration for each participant (= number of non-NaN frames)
- the **detected extrema** (notebook 4): to calculate QoM and
  Expansiveness

In [2]:
movements_path = DATA_DIR / "movement" / "03_normalized_signals.csv"
extrema_path = DATA_DIR / "movement" / "04_extrema.csv"

df_movements = pd.read_csv(movements_path)
df_extrema = pd.read_csv(extrema_path)

print(f"Movements : {df_movements.shape}")
print(f"Extrema   : {df_extrema.shape}")
print(f"\nParticipants : {df_movements['participant_id'].nunique()}")
display(df_extrema.head())

Movements : (36238, 9)
Extrema   : (3505, 6)

Participants : 40


,group,participant_id,signal,frame,type,value
0,P,P1,left_hand_vertical,2,min,-2.307631
1,P,P1,left_hand_vertical,41,max,-2.047892
2,P,P1,left_hand_vertical,69,min,-2.470514
3,P,P1,left_hand_vertical,108,max,-1.175648
4,P,P1,left_hand_vertical,122,min,-1.828153


## 5.3 Calculating Durations by (Participant, Signal)

The duration to be used for QoM_rate is the one during which the signal is
**actually available** (= no NaN). If a signal has been
rejected for that participant (all NaN), it will not appear in
the result.

We count the non-NaN frames for each (participant, signal)
and convert them to seconds.

In [3]:
def compute_durations(df_movements, signal_cols=MOVEMENT_SIGNALS, fps=FPS):
    """
    For each (participant, signal), return the number of
    non-NaN frames and the corresponding duration in seconds.
    """
    rows = []
    for pid, sub in df_movements.groupby("participant_id", sort=False):
        group = sub["group"].iloc[0]
        for sig_col in signal_cols:
            n_valid = int(sub[sig_col].notna().sum())
            rows.append({
                "group": group,
                "participant_id": pid,
                "signal": sig_col,
                "n_valid_frames": n_valid,
                "duration_sec": n_valid / fps,
            })
    return pd.DataFrame(rows)


durations = compute_durations(df_movements)
print("Durations per (participant, signal) :")
display(durations.head(10))

Durations per (participant, signal) :


,group,participant_id,signal,n_valid_frames,duration_sec
0,P,P1,left_hand_vertical,904,30.133333
1,P,P1,right_hand_vertical,904,30.133333
2,P,P1,left_hand_horizontal,904,30.133333
3,P,P1,right_hand_horizontal,904,30.133333
4,P,P1,head_horizontal,904,30.133333
5,P,P1,head_vertical,904,30.133333
6,P,P10,left_hand_vertical,915,30.500000
7,P,P10,right_hand_vertical,915,30.500000
8,P,P10,left_hand_horizontal,915,30.500000
9,P,P10,right_hand_horizontal,915,30.500000


## 5.4 Features per Signal

For each (participant, signal) pair, we calculate:

- `qom`: number of detected extrema (maxima + minima).
- `qom_rate`: `qom / duration_sec`, expressed as extrema per second.

**NaN vs. 0:**
- If `duration_sec > 0` but no extrema are detected, `qom = 0` and `qom_rate = 0`.
- If `duration_sec = 0`, the signal is missing/rejected and `qom_rate = NaN`.

In [4]:
def compute_features_per_signal(df_extrema, durations):
    """
    Compute QoM and QoM rate for each (participant, signal).

    - QoM: number of detected extrema.
    - QoM rate: number of extrema per second.
    - If the signal is present but no extrema are detected, QoM and QoM rate = 0.
    - If the signal is absent/rejected (duration = 0), QoM rate = NaN.
    """
    rows = []

    for _, dur_row in durations.iterrows():
        pid = dur_row["participant_id"]
        sig = dur_row["signal"]
        duration_sec = float(dur_row["duration_sec"])

        ext = df_extrema[
            (df_extrema["participant_id"] == pid)
            & (df_extrema["signal"] == sig)
        ]

        qom = len(ext)

        if duration_sec > 0:
            qom_rate = qom / duration_sec
        else:
            qom_rate = np.nan

        rows.append({
            "group": dur_row["group"],
            "participant_id": pid,
            "signal": sig,
            "n_valid_frames": int(dur_row["n_valid_frames"]),
            "duration_sec": duration_sec,
            "qom": qom,
            "qom_rate": qom_rate,
        })

    return pd.DataFrame(rows)


features_per_signal = compute_features_per_signal(df_extrema, durations)

print(f"Features per signal: {features_per_signal.shape}")
display(features_per_signal.head(10))

Features per signal: (240, 7)


,group,participant_id,signal,n_valid_frames,duration_sec,qom,qom_rate
0,P,P1,left_hand_vertical,904,30.133333,30,0.995575
1,P,P1,right_hand_vertical,904,30.133333,28,0.929204
2,P,P1,left_hand_horizontal,904,30.133333,25,0.829646
3,P,P1,right_hand_horizontal,904,30.133333,27,0.896018
4,P,P1,head_horizontal,904,30.133333,15,0.497788
5,P,P1,head_vertical,904,30.133333,10,0.331858
6,P,P10,left_hand_vertical,915,30.500000,24,0.786885
7,P,P10,right_hand_vertical,915,30.500000,25,0.819672
8,P,P10,left_hand_horizontal,915,30.500000,21,0.688525
9,P,P10,right_hand_horizontal,915,30.500000,24,0.786885


### 5.4.1 Overview of Features by Signal

Pivot table for a quick look at typical values.

In [5]:
for metric in ["qom", "qom_rate"]:
    pivot = features_per_signal.pivot(
        index="participant_id", columns="signal", values=metric,
    )
    print(f"\n=== {metric} ===")
    display(pivot.round(3))


=== qom ===


signal,head_horizontal,head_vertical,left_hand_horizontal,left_hand_vertical,right_hand_horizontal,right_hand_vertical
participant_id,,,,,,
P1,15,10,25,30,27,28
P10,11,19,21,24,24,25
P11,23,15,10,23,13,3
P12,15,13,34,39,29,37
P13,3,6,6,11,4,6
P14,6,12,31,26,20,16
P15,15,13,7,10,8,2
P16,15,24,37,26,31,28
P17,3,1,1,2,1,1



=== qom_rate ===


signal,head_horizontal,head_vertical,left_hand_horizontal,left_hand_vertical,right_hand_horizontal,right_hand_vertical
participant_id,,,,,,
P1,0.498,0.332,0.830,0.996,0.896,0.929
P10,0.361,0.623,0.689,0.787,0.787,0.820
P11,0.762,0.497,0.331,0.762,0.430,0.099
P12,0.496,0.430,1.123,1.289,0.958,1.222
P13,0.099,0.197,0.197,0.362,0.132,0.197
P14,0.195,0.390,1.008,0.845,0.650,0.520
P15,0.487,0.422,0.227,0.325,0.260,0.065
P16,0.487,0.779,1.201,0.844,1.006,0.909
P17,0.099,0.033,0.033,0.066,0.033,0.033


## 5.5 Aggregated Signals

We construct 2 **aggregated** signals. For each one, **the metrics** (`qom`,
`qom_rate`) are recalculated.

The aggregations:

- `hands_horizontal` = `left_hand_horizontal` + `right_hand_horizontal`
- `hands_vertical` = `left_hand_vertical` + `right_hand_vertical`

**Aggregation method.** Each metric is summed
independently of the others:
- Aggregated `qom` = sum of the `qom` values of the components

For duration, we take the minimum of the component durations
(= the window where **all** are valid). In practice, the
components have the same duration (same landmarks).

In [6]:
AGGREGATIONS = {
    "hands_horizontal": ["left_hand_horizontal", "right_hand_horizontal"],
    "hands_vertical": ["left_hand_vertical", "right_hand_vertical"]
}

def compute_features_aggregated(features_per_signal, aggregations=AGGREGATIONS):
    """
    Compute aggregated QoM features by summing the number of extrema
    across the corresponding component signals.

    If all component signals are valid, QoM rate is calculated per second.
    If at least one component is missing/rejected, QoM rate is set to NaN.
    """
    rows = []

    for agg_name, components in aggregations.items():
        sub = features_per_signal[
            features_per_signal["signal"].isin(components)
        ]

        for pid, group_sub in sub.groupby("participant_id", sort=False):
            group = group_sub["group"].iloc[0]
            duration_sec = float(group_sub["duration_sec"].min())
            n_valid_frames = int(group_sub["n_valid_frames"].min())

            qom_total = int(group_sub["qom"].sum())

            if duration_sec > 0:
                qom_rate = qom_total / duration_sec
            else:
                qom_rate = np.nan

            rows.append({
                "group": group,
                "participant_id": pid,
                "signal": agg_name,
                "n_valid_frames": n_valid_frames,
                "duration_sec": duration_sec,
                "qom": qom_total,
                "qom_rate": qom_rate,
            })

    return pd.DataFrame(rows)


features_aggregated = compute_features_aggregated(features_per_signal)

print(f"Aggregated features: {features_aggregated.shape}")
display(features_aggregated)

Aggregated features: (80, 7)


,group,participant_id,signal,n_valid_frames,duration_sec,qom,qom_rate
0,P,P1,hands_horizontal,904,30.133333,52,1.725664
1,P,P10,hands_horizontal,915,30.500000,45,1.475410
2,P,P11,hands_horizontal,906,30.200000,23,0.761589
3,P,P12,hands_horizontal,908,30.266667,63,2.081498
4,P,P13,hands_horizontal,912,30.400000,10,0.328947
...,...,...,...,...,...,...,...
75,SZ,SZ5,hands_vertical,924,30.800000,58,1.883117
76,SZ,SZ6,hands_vertical,907,30.233333,32,1.058434
77,SZ,SZ7,hands_vertical,906,30.200000,16,0.529801
78,SZ,SZ8,hands_vertical,908,30.266667,2,0.066079


## 5.7 Saving

A single output file: `05_qom.csv`
which contains **12 rows per participant**:
- 4 rows for the signals
  (`hands_horizontal`, `hands_vertical`, `head_vertical`, `head_horizontal`)

In [7]:
# Keep the two aggregated hand signals
hands_final = features_aggregated[
    features_aggregated["signal"].isin([
        "hands_horizontal",
        "hands_vertical",])]

# Keep the two head signals
head_final = features_per_signal[
    features_per_signal["signal"].isin([
        "head_horizontal",
        "head_vertical",])]

# Combine the four final signals
features_final = pd.concat(
    [hands_final, head_final],
    ignore_index=True,)

# Keep only the variables needed for subsequent analyses
features_final = features_final[
    ["group", "participant_id", "signal", "qom_rate"]]

out_path = DATA_DIR / "movement" / "05_qom.csv"
features_final.to_csv(out_path, index=False)

print(f"Saved -> {out_path}")
print(f"Shape: {features_final.shape}")
print(f"Signals: {sorted(features_final['signal'].unique())}")

display(features_final.head(12))

Saved -> C:\Users\factory\Desktop\pons.lea\data\movement\05_qom.csv
Shape: (160, 4)
Signals: ['hands_horizontal', 'hands_vertical', 'head_horizontal', 'head_vertical']


,group,participant_id,signal,qom_rate
0,P,P1,hands_horizontal,1.725664
1,P,P10,hands_horizontal,1.475410
2,P,P11,hands_horizontal,0.761589
3,P,P12,hands_horizontal,2.081498
4,P,P13,hands_horizontal,0.328947
5,P,P14,hands_horizontal,1.657638
6,P,P15,hands_horizontal,0.487013
7,P,P16,hands_horizontal,2.207792
8,P,P17,hands_horizontal,0.066298
9,P,P18,hands_horizontal,0.852459
